# سبق 04 - ٹول استعمال کرنے کا ڈیزائن پیٹرن

اس سبق میں آپ مائیکروسافٹ ایجنٹ فریم ورک (پائتھن) کا استعمال کرتے ہوئے AI ایجنٹس کے لیے **ٹول استعمال** ڈیزائن پیٹرن سیکھیں گے۔ ہم شامل کریں گے:

- `@tool` ڈیکوریٹر اور ٹائپ کیے گئے پیرامیٹرز کے ساتھ فنکشن ٹولز کی تعریف
- ٹول کے اسکیمے فراہم کرنا تاکہ ماڈل جان سکے کہ ہر ٹول کیا کرتا ہے
- `approval_mode` کے ذریعے ٹول کے اجرا کو کنٹرول کرنا
- Pydantic ماڈلز اور `response_format` کے ذریعے **منظم آؤٹ پٹ** واپس کرنا

منظر نامہ ایک **سفر بُکنگ ایجنٹ** ہے جو مقامات تلاش کر سکتا ہے، دستیابی چیک کر سکتا ہے، اور پرواز کی معلومات حاصل کر سکتا ہے۔


## ترتیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ڈیکوریٹر کے ساتھ ٹولز کی تعریف کرنا

`@tool` ڈیکوریٹر ایک سادہ پائتھن فنکشن کو ایک ایسا ٹول بنا دیتا ہے جسے ایک ایجنٹ کال کر سکتا ہے۔
اہم نکات:

- **docstring** ماڈل کی نظر میں ٹول کی وضاحت بن جاتی ہے۔
- **ٹائپ اینوٹیشنز** (جس میں تفصیلات کے ساتھ `Annotated` شامل ہے) ٹول اسکیمہ کی تعریف کرتے ہیں۔
- `approval_mode` کنٹرول کرتا ہے کہ آیا صارف کو ہر کال کی اجازت پہلے دینی ہوگی یا نہیں۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایک ایجنٹ بنانا جس میں متعدد آلات ہوں

تمام تین آلات کلائنٹ کو بھیجیں تاکہ ماڈل صارف کے سوال کا جواب دینے کے لیے جو بھی آلات درکار ہوں انہیں استعمال کر سکے۔


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## اوزاروں کے ساتھ منظم آؤٹ پٹ

`response_format` کو ایک Pydantic ماڈل پر سیٹ کرکے، ایجنٹ کو مجبور کیا جاتا ہے کہ وہ آزاد شکل کے متن کی بجائے ایک درست ٹائپ شدہ JSON آبجیکٹ واپس کرے۔ یہ اس وقت مفید ہوتا ہے جب نیچے والا کوڈ نتیجہ کو پروگراماتی طور پر استعمال کرنا چاہتا ہو۔


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ٹول منظوری کے پیٹرنز

`@tool` پر `approval_mode` پیرامیٹر کنٹرول کرتا ہے کہ آیا ٹول کالز کو اجرا کرنے سے پہلے انسانی منظوری درکار ہے:

| موڈ | برتاؤ |
|---|---|
| `"never_require"` | ٹول خود بخود چلتا ہے — صارف کی تصدیق ضروری نہیں۔ |
| `"always_require"` | ہر کال کو اجرا کرنے سے پہلے صارف کی منظوری لازمی ہے۔ |

ایسے ٹولز کے لیے `"always_require"` استعمال کریں جن کے ضمنی اثرات ہوتے ہیں (جیسے پرواز کی بکنگ، کریڈٹ کارڈ چارج کرنا) تاکہ انسان عمل میں شامل رہے۔


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **اوزار کو متعین کریں** `@tool` ڈیکوریٹر کا استعمال کرتے ہوئے جس میں ٹائپ شدہ پیرامیٹرز اور ڈاک سٹرنگز شامل ہوں جو اوزار کا خاکہ فراہم کریں۔
2. **متعدد اوزاروں کو ترتیب دیں** تاکہ ایجنٹ انہیں متسلسل طور پر کال کر کے پیچیدہ سوالات کے جواب دے سکے۔
3. **منظم آؤٹ پٹ واپس کریں** پائیڈینٹک ماڈل کو `response_format` کے طور پر پاس کر کے۔
4. **اوزار کی منظوری کو کنٹرول کریں** `approval_mode` کے ذریعے تاکہ حساس آپریشنز میں انسان کو شامل رکھا جا سکے۔

یہ طریقے قابل اعتماد، پروڈکشن کے لئے تیار ایجنٹس بنانے کی بنیاد فراہم کرتے ہیں جو بیرونی نظاموں کے ساتھ حفاظت سے تعامل کر سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
